# DLB Workflow #

This section presents how to go through the full procedure of calculating the load envelopes for a turbine model according to IEC61400-1 using `wetb`.

## DLB definition ##

The first step is to define the cases to be simulated. For doing so, we can make an instance of the `DTU_IEC61400_1_Ref_DLB` class with our turbine-specific variables:

- `iec_wt_class`: wind turbine class according to IEC61400-1. It is used to determine Vref and Iref, which in turn are used in the calculation of TI (NTM, ETM), gusts (EOG, EDC) and shear profile (EWS).
- `D`: rotor diameter in meters. Used in EOG, EDC, EWS.
- `z_hub`: hub height in meters. Used to calculate $\lambda_1$, which in turn is used in EOG, EDC, EWS.
- `Vin`: cut-in wind speed.
- `Vout`: cut-in wind speed.
- `Vr`: cut-in wind speed.

For site-specific load assessments, the user can pass the following optional arguments:
- `alpha`: exponent of power law. Used in NWP and EWS. The default is 0.2.
- `alpha_extreme`: exponent of power law in extreme wind. Used in NWP for DLCs 6.1, 6.2, 6.3 and 7.1. The default is 0.11.
- `ti_extreme`: TI for DLCs with extreme wind. Used for DLCs 6.1, 6.2, 6.3 and 7.1. The default is 0.11.
- `Vstep`: step in simulated wind speed. The default is 2 m/s.

If the user uses custom DLLs for the controller, generator servo and pitch servo instead of the DTU ones the following arguments need to be passed:
- `controller`: name of the controller dll block in the htc file. Needed for setting the operational mode. The default is 'dtu_we_controller'
- `generator_servo`: name of the generator servo dll block in the htc file. Needed for grid loss (DLCs 2.1, 2.3). The default is 'generator_servo'.
- `pitch_servo`: name of the pitch servo dll block in the htc file. Needed for stuck blade (DLC 2.2b) and pitch runaway (DLC 2.2p, should not be simulated). The default is 'servo_with_limits'.
- `constant_cutin`: number of the constant in the controller dll that corresponds to the cut-in time. The default is 24.
- `constant_cutout`: number of the constant in the controller dll that corresponds to the cut-out time. The default is 26.
- `constant_shutdown_type`: number of the constant in the controller dll that corresponds to the shut-down type. The default is 28.
- `constant_gridloss_time`: number of the constant in the generator servo dll that corresponds to the grid loss time. The default is 7.
- `constant_stuckblade_time`: number of the constant in the pitch servo dll that corresponds to the blade stuck time. The default is 9.
- `constant_stuckblade_angle`: number of the constant in the pitch servo dll that corresponds to the blade stuck angle. The default is 10.
- `constant_pitchrunaway_time`: number of the constant in the controller dll that corresponds to the pitch runaway time. The default is 8.

Finally if the user wants to simulate DLCs 7.1 and 8.1 (which should not be run due to structural convergence issues) the following arguments need to be passed:
- `shaft_mbdy`: name of the shaft main body in the htc file. Needed for locking the rotor (DLCs 7.1, 8.1). The default is 'shaft'.
- `shaft_constraint`: name of the shaft constraint in the htc file. Needed for locking the rotor (DLCs 7.1, 8.1) The default is 'shaft_rot'.
- `Vmaint`: maintenance wind speed. Used in DLC 8.1. The default is 18 m/s.
- `best_azimuth`: best azimuthal position degrees in which the rotor is locked for DLC 8.1. The default is 180 degrees.

In [ ]:
from wetb.dlb.iec61400_1 import DTU_IEC61400_1_Ref_DLB

dlb = DTU_IEC61400_1_Ref_DLB(iec_wt_class='1B',
                             D=240,
                             z_hub=150,
                             Vin=3,
                             Vr=10,
                             Vout=25)

DLCs 2.2p should not be run since most turbines have safety systems that shut down the turbine in the event of a pitch runaway. DLCs 7.1 and 8.1 can be run, but the user should be aware that structural convergence might not be achieved when the rotor is locked and that VIV may occur at very high angles of attack, which is not modelled in HAWC2 (an FSI simulation would be necessary, i.e. HAWC2 + EllipSys). So the user can remove them from the DLB by modifying the `dlcs` attribute:

In [25]:
dlb.dlcs.drop(['DLC22p', 'DLC71', 'DLC81'], inplace=True)

## Input file writing ##

The next step is to generate the htc files for all cases. For doing so, the user can create an instance of the `HAWC2InputWriter` class from the existing instance of `DTU_IEC61400_1_Ref_DLB` through `from_pandas` method, passing also the path to the template htc file and the turbine diameter (necessary for the turbulence box size). Then the user can write the htc files with the `write_all` method:

In [5]:
from wetb.dlb.hawc2_iec_dlc_writer import HAWC2_IEC_DLC_Writer

h2writer = HAWC2_IEC_DLC_Writer(base_htc_file='../../tutorials/data/htc/IEA_15MW_RWT.htc',
                                diameter=240).from_pandas(dlb)
h2writer.write_all(out_dir='.../../tutorials/data/htc/DLB')

Generating 1472 htc files in directory: ...\..\tutorials\data\htc\DLB


100%|██████████| 1472/1472 [04:38<00:00,  5.29it/s]


`HAWC2_IEC_DLC_Writer` has optional arguments `start_time` and `turbulence_defaults` that might need to be passed if the user needs differents values from the default ones.
- `start_time`: defaults to 100s. It is usually fine for most turbines, but larger values might be necessary for larger turbines.
- `turbulence_defaults`: defaults to [33.6, 3.9, 8192, 64] ($L, \gamma, n_x, n_{yz}$). $L$ follows the IEC61400-1 for turbines with $z_{hub} > 60 m$, $\gamma$ corresponds to neutral atmospheric conditions and 8192 and 64 are powers of 2 that show a good compromise between box size and accuracy.

## Running HAWC2 ##

The next step is to run HAWC2 for all htc files. Running HAWC2 using `wetb` has already been explained in a previous section. The user should just be aware that due to the high number of simulations within a full DLB, running simulations serially is not feasible. So the user is expected to have computational resources for running several occurrences of HAWC2 concurrently i.e. in a cluster.

## Sanity checks ##

Once all simulations have finished running, the user should check that they all have finished successfully. That means looking into the log files to confirm they reached the end time without errors, and perhaps also looking into specific time series. A good way to identify potentially troublesome simulations (load outliers, accidental shut-downs...) is to look at the post-processed statistics of the time series, as it is not feasible to look into the time series of every individual simulation.

## Time series post-processing ##

Once the user is confident that all simulations outputs are correct they can be post-processed. The post-processing of multiple .hdf5 files was explained in a previous section, using the `collect_postproc` function. For the general case of a DLB, the minimum set of post-processing functions is the following:
- `statistics`: always useful for performing sanity checks.
- `extreme_loads`: computes the extreme load table according to IEC61400-1 for the specified load sensors. The user must specify the name and 6 indices of the (Fx, Fy, Fz, Mx, My, Mz) components of each load of interest through the `sensors_info` argument.
- `equivalent_loads`: computes the 1 Hz Damage Equivalent Loads of all output sensors for a given set of Wöhler slopes.

These functions are already available in `wetb.utils.postprocs`:

In [ ]:
from wetb.utils.postprocs import (statistics,
                                  extreme_loads,
                                  equivalent_loads)
from wetb.gtsdf.gtsdf import collect_postproc

sensors_info = (['Tower base', 15, 16, 17, 18, 19, 20],
                ['Yaw bearing', 69, 70, 71, 72, 73, 74],
                ['Main bearing', 75, 76, 77, 78, 79, 80],
                ['Blade 1 root', 81, 82, 83, 84, 85, 86],
                ['Blade 2 root', 285, 286, 287, 288, 289, 290],
                ['Blade 3 root', 489, 490, 491, 492, 493, 494])

config = {statistics: {},
          extreme_loads: {'sensors_info': sensors_info},
          equivalent_loads: {}}

statistics, extreme_loads, equivalent_loads = collect_postproc(folder='../../tutorials/data/res',
                                                               config=config)

After calculation, auxiliar coordinates with the simulation parameters (`dlc`, `wsp`, `wdir`) must be added. This is done by `add_coords_from_filename`, as explained in previous sections:

In [36]:
from wetb.dlb.dlb_postprocs import add_coords_from_filename

statistics = add_coords_from_filename(dataarray=statistics,
                                      params=['dlc', 'wsp', 'wdir'],
                                      regex=r'DLC(\w+?)_wsp(\d{2})_wdir(\d{3})',
                                      formats=[str, float, float])
print(statistics)

<xarray.DataArray (filename: 1472, sensor_name: 758, statistic: 4)> Size: 36MB
array([[[ 3.12611065e-03,  1.79180923e+02,  3.59986847e+02,
          1.03999947e+02],
        [ 4.66623068e+00,  4.93828249e+00,  5.09246492e+00,
          1.21376120e-01],
        [ 1.18784264e-04,  2.00660944e+00,  2.60820436e+00,
          8.33479881e-01],
        ...,
        [ 2.38915504e-06,  3.50211896e-02,  4.55210358e-02,
          1.45465229e-02],
        [ 2.23695451e-06,  3.50211710e-02,  4.55213226e-02,
          1.45464921e-02],
        [ 1.44532471e+01,  2.31393890e+01,  3.40624695e+01,
          6.30237675e+00]],

       [[ 6.50851848e-03,  1.79911789e+02,  3.59997681e+02,
          1.04299988e+02],
        [ 4.99486780e+00,  5.05117416e+00,  5.09685564e+00,
          1.63158961e-02],
        [ 0.00000000e+00,  2.10847306e+00,  2.60855460e+00,
          7.27039635e-01],
...
        [ 1.57079637e+00,  1.57035494e+00,  1.57079637e+00,
          4.41274606e-04],
        [ 1.57079637e+00,  1.570

In [37]:
extreme_loads = add_coords_from_filename(dataarray=extreme_loads,
                                         params=['dlc', 'wsp', 'wdir'],
                                         regex=r'DLC(\w+?)_wsp(\d{2})_wdir(\d{3})',
                                         formats=[str, float, float])
print(extreme_loads)

<xarray.DataArray (filename: 1472, sensor_name: 6, driver: 14, load: 6)> Size: 6MB
array([[[[ 7.05022766e+02,  4.25653381e+01, -1.76517715e+04,
           6.44703203e+04,  8.44967031e+04, -2.81193799e+03],
         [-6.91995178e+02,  4.99744965e+02, -1.76665781e+04,
           3.76464209e+03, -8.32940312e+04, -8.84857544e+02],
         [-1.45440048e+02,  1.16799695e+03, -1.76712715e+04,
          -6.83470078e+04, -1.64816035e+04,  3.90332056e+03],
         ...,
         [ 1.82680492e+01,  2.43363403e+02, -1.75462637e+04,
           3.89681836e+04, -4.15363330e+03, -7.35613623e+03],
         [-1.80748718e+02,  1.16520569e+03, -1.76711133e+04,
          -6.85395156e+04, -1.88871621e+04,  4.20140088e+03],
         [ 7.57857513e+01, -8.87205566e+02, -1.76127305e+04,
           1.78407172e+05,  9.33510938e+03,  3.07981616e+03]],

        [[ 5.84009888e+02, -1.34032410e+02, -9.30980859e+03,
           6.59066484e+04,  2.60233569e+03,  2.02830347e+03],
         [-5.74588745e+02,  5.78513794e+

The equivalent loads are computed for all output sensors, but the user can slice them to only keep the loads of interest, i.e. the ones from `sensors_info`:

In [38]:
equivalent_loads = add_coords_from_filename(dataarray=equivalent_loads,
                                            params=['dlc', 'wsp', 'wdir'],
                                            regex=r'DLC(\w+?)_wsp(\d{2})_wdir(\d{3})',
                                            formats=[str, float, float])

load_indices = [s[i] for s in sensors_info for i in range(1, 7)]
equivalent_loads = equivalent_loads.isel(sensor_name=load_indices)
print(equivalent_loads)

<xarray.DataArray (filename: 1472, sensor_name: 36, m: 6)> Size: 3MB
array([[[  426.83144323,   528.80509244,   681.15629337,   786.22905576,
           862.19463463,   919.65995521],
        [  738.32055481,   874.41861734,  1063.75874547,  1194.50722718,
          1292.68061289,  1369.78605486],
        [  167.35806601,   176.99482589,   195.54342401,   210.72430771,
           223.17191486,   233.70042519],
        ...,
        [ 7303.91665789,  9114.71664064, 11675.35693232, 13546.40391875,
         15084.84999784, 16397.87526912],
        [15279.97396691, 18817.05041834, 23179.82484307, 25734.48084599,
         27406.6848298 , 28586.66145447],
        [  297.26075296,   369.70031459,   465.74609397,   529.81863881,
           578.72934187,   619.03696903]],

       [[  335.55459181,   398.0487286 ,   496.86864501,   572.95824534,
           633.36930847,   681.93801389],
        [  594.53409879,   693.07638491,   827.56719368,   919.45465497,
           989.10426398,  1045.1500838

## DLB post-processing ##

After the time series have been post-processed, the user can perform the post-processings at DLB level. Such post-processing functions are available in `wetb.dlb.dlb_postprocs` and take in collected `xarray.DataArray` output by the time series post-processing functions, returning a `xarray.DataArray` where the `filename` dimension has been squeezed into a single value for all simulations. The most important ones are `get_DLB_extreme_loads` and `get_DLB_equivalent_loads`.

### Extreme loads ###

The extreme load table defined in IEC61400-1 consists of the (Fx, Fy, Fz, Mx, My, Mz) combination for each case where one of the components is maximum or minimum within a time series (the other 5 components correspond to the contemporaneous values at the instant it occurs) as well as the 2 cases where the resulting F and M are maximum (so 14 cases in total).
To assign a single value to the DLB from all simulations, simulations go through the following process:
- Binning: simulations are grouped into bins that share the same features (usually DLC and wind speed). This is specified through the `regex_list` argument, a dictionary that assigns a regular expression to each DLC. The default binning is DLC and wind speed and is consistent with the naming from `DTU_IEC61400_1_Ref_DLB`, so in general the user will not need to pass this argument.
- Metric application: a metric is applied to the values of each bin, assigning a single value to each bin. This is specified through the `metric_list` argument, a dictionary that assigns a function (`mean`, `max`, `upperhalf`...) to each DLC. The default metrics follow the IEC61400-1, so in general the user will not need to pass this argument.
- Safety factor application: the metric values are multiplied by some safety factor that depends on the DLC. This is specified through the `safety_factor_list` argument. The default safety factors follow the IEC61400-1, so in general the user will not need to pass this argument.

Finally the maximum or minimum value (depending on whether the driving load component is a maximum or a minimum) of all the bins is selected for that load component. For the contemporaneous load components, its calculation will depend on the `contemporaneous_method` argument:
- `averaging` (default): the absolute value of each load component is averaged across the different cases in the bin, and the sign corresponds to the sign of the load component from the case with the highest load. 
- `scaling`: a scaling factor is computed by dividing the bin metric value by the value of the case with the closest value, and then the contemporaneous loads of such a case are multiplied by this scaling factor.

In [39]:
from wetb.dlb.dlb_postprocs import get_DLB_extreme_loads

DLB_extreme_loads = get_DLB_extreme_loads(extreme_loads)
print(DLB_extreme_loads)

<xarray.DataArray (sensor_name: 6, driver: 14, load: 10)> Size: 7kB
array([[[ 8.04531006e+03, -1.44502512e+03, -1.76393415e+04,
          2.18346406e+05,  3.43486681e+05, -2.81232257e+04,
          8.17405111e+03, -1.01823763e+01,  4.07011368e+05,
          5.75568392e+01],
        [-1.74079137e+03,  8.06473890e+02, -1.77179430e+04,
         -7.36567740e+04, -1.43218419e+05,  9.67451389e+03,
          1.91852932e+03,  1.55142652e+02,  1.61049172e+05,
         -1.17216614e+02],
        [-1.69981556e+03,  6.49371884e+03, -1.76411790e+04,
         -2.83643996e+05, -2.02070587e+05, -2.11478227e+04,
          6.71250753e+03,  1.04668794e+02,  3.48262026e+05,
         -1.44533579e+02],
        [-2.65781259e+02, -2.86346666e+03, -1.75840688e+04,
          3.16727594e+05, -3.43249645e+04, -8.84047887e+03,
          2.87577485e+03, -9.53028863e+01,  3.18582127e+05,
         -6.18522033e+00],
        [-1.79789384e+03,  1.00140334e+03, -1.45882430e+04,
         -1.33432744e+05, -2.02123809e+05,  

## Equivalent loads ##

The damage equivalent loads of the DLB are computed by `get_DLB_eq_loads`, which takes in the damage equivalent loads of each individual time series and aggregates them all through Palmgren-Miner rule. Since the simulation time of each time series is different to its actual time within the total lifetime of the turbine, the weights (real time / simulation time) of each individual time series need to be calculated.

### Weight list calculation ###

The function `get_weight_list` computes the weights for all time series included in fatigue analysis. It has several keyword and optional arguments that allow different levels of customization, from a very simple use with only keyword arguments to more complex setups for site-specific cases:

#### Keyword arguments ####
- `dataarray`: `xarray.DataArray` containing auxiliar coordinates `filename` and every other conditioned variable (`wsp`, `wdir`, `dlc`...). The collected output from `equivalent_loads` should be passed.
- `n_years`: lifetime of the turbine in years.
- `Vin`: cut-in wind speed.
- `Vout`: cut-out wind speed.
- `Vr`: rated wind speed.
- `Vref`: reference wind speed.
- `Vstep`: step in simulated wind speeds.

#### Optional arguments ####
- `wsp_dist`: `xarray.DataArray` containing the distribution of `wsp`. This argument allows site-specific load assessments. If not passed, the default corresponds to a monodirectional Weibull distribution according to IEC61400-1.
- `dlc_dist`: `xarray.DataArray` containing the joint distribution of `dlc` w.r.t. `wsp`. If not passed, the default assumes that the turbine is always idling outside of the operational wind speed range, and `weight_DLC64_Vin_Vout` of the time inside that range. The turbine is in normal power production for the rest of the time inside the operational wind speed range, most of which is assigned to DLC 12 and only 50 hours per year to DLC 24.
- `yaw_dist`: `xarray.DataArray` containing the joint distribution of `wdir` w.r.t. `dlc`. If not passed, the default assumes that for DLC 12 the turbine is aligned with the wind 50% of the time and 25% for -10°/10° yaw misalignments respectively, whereas for DLCs 24 and 64 it's 50% for -20°/20° and -8°/8° respectively.
- `weight_DLC64_Vin_Vout`: percentage of the time the turbine is idling in the operational wind speed range (only used if `dlc_dist` is not passed). The default is 2.5%, but the user might consider a different value. 
- `n_seeds`: `xarray.DataArray` containing the number of seeds per DLC for DLCs 12, 24, 64. The default follows the IEC61400-1.
- `n_events`: `xarray.DataArray` containing the number of events per DLC and wind speed for DLCs 31 and 41. The default follows the IEC61400-1.
- `sim_time`: `xarray.DataArray` containing the simulation time per DLC. The default follows the IEC61400-1.

`n_seeds` and `sim_time` should only be passed in the very unlikely event of simulating extra seeds or a different simulation time, and `yaw_dist` and `n_events` if the standard ever changes. `wsp_dist` and `dlc_dist` or `weight_DLC64_Vin_Vout` are what the user might need to pass in realistic custom scenarios.

Note that currently `wdir` corresponds to yaw misalignment, as only yaw misalignments are simulated (just like `wavedir` would correspond to wind-wave misalignment in offshore turbines). This is done in order to reduce the number of simulations. For loads of rotor-nacelle components this is fine as they rotate, but for loads of the support structure the actual `wdir` and `wavedir` are not the same. `wetb` does not currently support this, but the current approach is conservative as it assumes that the wind is always coming from the same direction.

Below is an example of the most simple case, only using keyword arguments:

In [40]:
from wetb.dlb.dlb_postprocs import get_weight_list

n_years = 25
Vin = 3
Vr = 10
Vout = 25
Vref = 50
Vstep = 2

weight_list = get_weight_list(equivalent_loads, n_years, Vin, Vout, Vr, Vref, Vstep) # Only keyword arguments
print(weight_list)

<xarray.DataArray (filename: 498)> Size: 4kB
array([9.31190504e+03, 9.31190504e+03, 9.31190504e+03, 9.31190504e+03,
       9.31190504e+03, 9.31190504e+03, 4.65595252e+03, 4.65595252e+03,
       4.65595252e+03, 4.65595252e+03, 4.65595252e+03, 4.65595252e+03,
       4.65595252e+03, 4.65595252e+03, 4.65595252e+03, 4.65595252e+03,
       4.65595252e+03, 4.65595252e+03, 1.36961312e+04, 1.36961312e+04,
       1.36961312e+04, 1.36961312e+04, 1.36961312e+04, 1.36961312e+04,
       6.84806561e+03, 6.84806561e+03, 6.84806561e+03, 6.84806561e+03,
       6.84806561e+03, 6.84806561e+03, 6.84806561e+03, 6.84806561e+03,
       6.84806561e+03, 6.84806561e+03, 6.84806561e+03, 6.84806561e+03,
       1.58961420e+04, 1.58961420e+04, 1.58961420e+04, 1.58961420e+04,
       1.58961420e+04, 1.58961420e+04, 7.94807100e+03, 7.94807100e+03,
       7.94807100e+03, 7.94807100e+03, 7.94807100e+03, 7.94807100e+03,
       7.94807100e+03, 7.94807100e+03, 7.94807100e+03, 7.94807100e+03,
       7.94807100e+03, 7.9480710

### Custom case ###
For site-specific calculations, the user can pass `wsp_dist`. Below is an example of a custom Weibull distribution:

In [ ]:
import numpy as np
import xarray as xr
from wetb.dlc.high_level import Weibull2

wsp_list = np.unique(equivalent_loads.wsp.data)
u = 12 # Weibull scale parameters
k = 1.4 # Weibull shape parameters
wsp_dist = xr.DataArray(data=[p for p in Weibull2(u, k, wsp_list).values()],
                        dims=['wsp'],
                        coords={'wsp': wsp_list})

weight_list = get_weight_list(equivalent_loads, n_years, Vin, Vout, Vr, Vref, Vstep, wsp_dist=wsp_dist)

Furthermore, if the user wishes to change the percentage of time DLCs 12, 24 and 64 take `dlc_dist` or `weight_DLC64_Vin_Vout` can be passed. The most realistic scenario is changing the time the turbine will be idling in the operational range:

In [ ]:
weight_list = get_weight_list(equivalent_loads, n_years, Vin, Vout, Vr, Vref, Vstep, wsp_dist=wsp_dist, weight_DLC64_Vin_Vout=0.1)

In the unlikely event of an even more general case that is not supported by `get_weight_list` the user should define the weight_list himself in the same format, as a `xarray.DataArray` that assigns a weight (real time / simulation time) for each simulation.

Once the weights have been computed, then calling `get_DLB_eq_loads` is straightforward:

In [41]:
from wetb.dlb.dlb_postprocs import get_DLB_eq_loads

DLB_equivalent_loads = get_DLB_eq_loads(equivalent_loads, weight_list)
print(DLB_equivalent_loads)

<xarray.DataArray (sensor_name: 36, m: 6)> Size: 2kB
array([[  2409.19989314,   1898.36053283,   1635.61412351,
          1611.18873787,   1642.46662134,   1688.36053633],
       [  3169.05093897,   2546.25452853,   2207.86387162,
          2160.35305239,   2185.2349008 ,   2231.0243942 ],
       [  2558.16057251,   1935.40996788,   1580.71700763,
          1516.77191676,   1534.15637143,   1579.31362651],
       [302387.56740474, 263914.31287411, 248066.21704411,
        251307.28666558, 258718.73035692, 266867.17202402],
       [207395.84339215, 182959.83669398, 176231.91279614,
        181427.24562156, 188456.36585019, 195250.08096575],
       [ 56981.64345843,  48627.27041677,  47362.78922744,
         51037.96182036,  55426.02587983,  59631.56874576],
       [  1483.04230468,   1278.88519742,   1215.4125348 ,
          1247.24521501,   1294.5250897 ,   1341.22941045],
       [  2102.70935701,   1823.8354441 ,   1703.54317527,
          1720.77553692,   1768.43408493,   1822.319059

## Report generation ##

Finally Excel reports can be generated for both extreme and fatigue load envelopes:

In [43]:
from wetb.dlb.dlb_reporting import (DLB_extreme_loads_to_excel,
                                    dataarray_to_excel)

DLB_extreme_loads_to_excel(DLB_extreme_loads, '../../tutorials/data/reports/DLB_extreme_loads.xlsx')
dataarray_to_excel(DLB_equivalent_loads, '../../tutorials/data/reports/DLB_equivalent_loads.xlsx')

Additionally the `xarray.DataArray` variables can be saved into .nc files with `to_netcdf` method. This will be useful for further manipulation of the DLB loads in the future (i.e. for comparing loads between different models) without having to recompute them.

In [44]:
from wetb.dlb.dlb_postprocs import nc_to_dataarray

DLB_extreme_loads.to_netcdf('../../tutorials/data/postproc/DLB_extreme_loads.nc')
DLB_equivalent_loads.to_netcdf('../../tutorials/data/postproc/DLB_equivalent_loads.nc')